# ClinHallu: Whisper large-v3 / turbo evaluation (Kaggle or Colab, T4)
**Kaggle:** Settings → Accelerator **GPU T4**, Internet **On**. **Colab:** Runtime → T4 GPU.
Every step checks for errors and stops with a message if something fails.

In [ ]:
import os, subprocess, sys
def sh(cmd):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-3000:]); print(r.stderr[-3000:], file=sys.stderr)
    if r.returncode != 0: raise RuntimeError(f'FAILED ({r.returncode}): {cmd}')
WORK = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
os.chdir(WORK)
sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
if not os.path.exists('whisper-clinical-hallucination'):
    sh('git clone -q https://github.com/hamzak-27/whisper-clinical-hallucination.git')
os.chdir('whisper-clinical-hallucination'); sh('git log --oneline | head -3')
sh('pip -q install faster-whisper jiwer kokoro-onnx soundfile nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"')

In [ ]:
# Make the pip-installed CUDA libs visible to CTranslate2 (fixes 'libcudnn/libcublas not found')
import glob, site
libdirs = [os.path.dirname(p) for sp in site.getsitepackages() for p in glob.glob(sp + '/nvidia/*/lib/*.so*')]
print('CUDA lib dirs:', sorted(set(libdirs)))
os.environ['LD_LIBRARY_PATH'] = ':'.join(sorted(set(libdirs))) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
# GPU smoke test BEFORE spending time on data
sh('python -c "import numpy as np; from faster_whisper import WhisperModel; '
   'm=WhisperModel('tiny', device='cuda', compute_type='float16'); '
   'print(list(m.transcribe(np.zeros(16000*3, dtype=np.float32))[0])); print('GPU OK')"')

In [ ]:
sh('mkdir -p models/kokoro data/raw/esc50')
sh('wget -q -O models/kokoro/kokoro-v1.0.onnx https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx')
sh('wget -q -O models/kokoro/voices-v1.0.bin https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin')
sh('wget -q -O /tmp/esc50.zip https://github.com/karolpiczak/ESC-50/archive/master.zip && unzip -q -o /tmp/esc50.zip -d data/raw/esc50')

In [ ]:
# Build the pilot split (same seeds as the local run, about 3-5 min on CPU)
sh('cd src && python build_audio.py --split pilot --n 4 --gaps 1,12')
sh('wc -l data/clinhallu/pilot/manifest.jsonl')  # expect 104

In [ ]:
# Evaluate, most important first. New result files go to results/pilot/ next to the cloned CPU results.
RUNS = [('large-v3','default'), ('large-v3','no_prev'), ('large-v3-turbo','default'), ('large-v3-turbo','no_prev')]
for m, p in RUNS:
    sh(f'cd src && python run_asr.py --split pilot --model {m} --preset {p} --device cuda --compute float16')
    sh(f'wc -l results/pilot/{m}__{p}.jsonl')

In [ ]:
sh('cd src && python aggregate.py pilot | head -30')
sh(f'zip -qr {WORK}/results_gpu.zip results')
print('Download:', f'{WORK}/results_gpu.zip', '(Kaggle: Output panel; Colab: Files panel)')